# Introduction to NLP — Mini Project: Trigram Language Model and NER

Welcome to the mini project for our intro to Natural Language
Processing course!  In this assignment you will build a simple
trigram language model (LM) and use it as a feature generator for a
Named Entity Recognition (NER) classifier.  The objective is to
give you hands-on experience with n-gram language modelling,
feature engineering and a standard machine learning workflow.  You
will practise coding up a trigram LM with add-k smoothing, extract
features from it, evaluate a baseline NER system and experiment
with different smoothing parameters.

## Dataset

We provide a small dataset of sentences with manually annotated
named-entity labels.  The data comes from news articles and is
split into three parts:

* **Training set** (`ner_train.tsv`) – used to train the language
  model and the NER classifier.
* **Development set** (`ner_dev.tsv`) – used to tune the smoothing
  parameter for the trigram LM and to perform preliminary
  evaluation.
* **Test set** (`ner_test.tsv`) – held out for final evaluation.

Each file is a tab separated file with two columns.  The first
column contains a token (word or punctuation) and the second
column contains the corresponding BIO-style NER tag (O, B-LOC,
I-LOC and so on).  Blank lines separate sentences.  An example
from the training set looks like this:

    In	O
    Singapore	B-LOC
    ,	O
    UNICEF	B-ORG
    unveiled	O
    ...

Your first step is to load these files and organise them into a
list of sentences.  A helper function is provided in the next
cell.

In [28]:
import os
from typing import List, Tuple

def load_ner_data(path: str) -> List[Tuple[List[str], List[str]]]:
    """Load a TSV file into a list of token/tag sentences.

    Parameters
    ----------
    path : str
        Path to a TSV file with one token and one tag per line.
        Blank lines separate sentences.

    Returns
    -------
    List[Tuple[List[str], List[str]]]
        A list where each element is a tuple (tokens, tags) for a
        sentence.
    """
    sentences = []
    tokens: List[str] = []
    tags: List[str] = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == '':
                if tokens:
                    sentences.append((tokens, tags))
                    tokens, tags = [], []
            else:
                parts = line.split('	')
                if len(parts) != 2:
                    raise ValueError(f'Invalid line: {line}')
                tok, tag = parts
                tokens.append(tok)
                tags.append(tag)
    if tokens:
        sentences.append((tokens, tags))
    return sentences

# Load the data (assuming the TSV files are in the same directory).
train_data = load_ner_data('ner_train.tsv')
dev_data = load_ner_data('ner_dev.tsv')
test_data = load_ner_data('ner_test.tsv')
print(f'Loaded {len(train_data)} train, {len(dev_data)} dev and {len(test_data)} test sentences.')

Loaded 1200 train, 150 dev and 150 test sentences.


## Trigram Language Model

A language model assigns probabilities to word sequences. In a trigram LM, each word depends only on the previous two words:
$$
P(w_i \mid w_{i-2}, w_{i-1}).
$$

We estimate this from counts of trigrams \((w_{i-2}, w_{i-1}, w_i)\) and use **add-\(k\)** (Lidstone) smoothing to handle unseen events:
$$
P(w \mid u,v)
= \frac{\mathrm{count}(u,v,w)+k}
       {\sum_{w' \in V}\mathrm{count}(u,v,w') + k\,|V|},
$$
where \(V\) is the vocabulary and \(k \ge 0\). Setting \(k=0\) recovers the maximum-likelihood estimate; \(k=1\) is add-one (Laplace) smoothing. Larger \(k\) shifts probability mass from seen trigrams to unseen ones.


### Task 1: Implement the trigram LM

The class skeleton below collects n-gram counts in its `train`
method.  Your job is to finish the `next_word_logprobs` method
to compute log probabilities for each word in the vocabulary given a
two-word context.  Use add-k smoothing as defined above.  The
method should return a dictionary mapping each word to its
log probability (natural log).  Be sure to include all words in
the vocabulary, even those with very low counts, so that the
resulting vector forms a proper distribution.

In [34]:
from collections import Counter
import math
import numpy as np
from typing import Dict, Tuple, List

class TrigramLanguageModel:
    """A trigram language model with add-k smoothing.

    Parameters
    ----------
    k : float
        Smoothing constant.  Must be non-negative.
    """
    def __init__(self, k: float = 0.1):
        self.k = k
        self.unigram_counts: Counter = Counter()
        self.bigram_counts: Counter = Counter()
        self.trigram_counts: Counter = Counter()
        self.vocab: set[str] = set()

    def train(self, sentences: List[List[str]]) -> None:
        """
        TODO: Collect n-gram counts and build vocab.

        For each sentence `tokens` in `sentences`:
          1. Pad with two start tokens and one end token:
                padded = ['<s>', '<s>'] + tokens + ['</s>']
          2. Update the vocabulary with **only** the original tokens
             (special tokens are added after the loop):
                self.vocab.update(tokens)
          3. For every position i from 2 .. len(padded)-1:
                u, v, w = padded[i-2], padded[i-1], padded[i]
                self.trigram_counts[(u, v, w)] += 1
                self.bigram_counts[(u, v)] += 1
                self.unigram_counts[w] += 1   # counts of the next word (w)
        After processing all sentences, add the special tokens to the vocab:
            self.vocab.add('<s>'); self.vocab.add('</s>')

        Example (two sents: ['the','cat','sat'], ['the','dog','sat']):
        vocab ⊇ {'the','cat','dog','sat','<s>','</s>'}
        trigram[('the','cat','sat')] == 1
        bigram[('sat','</s>')] == 0
        unigram['</s>'] == 2
        """
        for tokens in sentences:
            # Pad with 2 start tokens and 1 end token
            padded = ['<s>', '<s>'] + tokens + ['</s>']
            # Update vocabulary with only original tokens
            self.vocab.update(tokens)
            for i in range(2, len(padded)):
                u, v, w = padded[i-2], padded[i-1], padded[i]
                # Update counts for trigrams, bigrams, unigrams
                self.trigram_counts[(u, v, w)] += 1
                self.bigram_counts[(u, v)] += 1
                self.unigram_counts[w] += 1   # Counts of next word (w)

        # After processing all sentences, add special tokens to vocab
        self.vocab.add('<s>')
        self.vocab.add('</s>')

    def next_word_logprobs(self, context: Tuple[str, str]) -> Dict[str, float]:
        """Return log probabilities for each word in the vocab given a context.

        Parameters
        ----------
        context : tuple[str, str]
            A tuple (u, v) of the two preceding words.

        Returns
        -------
        Dict[str, float]
            Mapping from word to log probability.

        TODO: implement smoothed probability calculation and replace
        the NotImplementedError.  Use add-k smoothing: for each w
        in vocab,

            P(w | u, v) = (count(u, v, w) + k) / (count(u, v, *) + k*|V|)

        Then take the natural log.
        """
        u, v = context
        vocab_size = len(self.vocab)
        # Count(u, v, *) = bigram count (u, v)
        context_count = self.bigram_counts[(u, v)]
        
        # Denominator: sum over w' (count(u, v, w') + k) = count(u, v, *) + k * |V|
        denominator = context_count + (self.k * vocab_size)
        
        logprobs = {}
        for w in self.vocab:
            # Numerator: count(u, v, w) + k
            trigram_count = self.trigram_counts[(u, v, w)]
            numerator = trigram_count + self.k
            
            # Calculate prob and nat log
            # Handle cases with 0
            if numerator == 0 or denominator == 0:
                # Placeholder: large negative number
                logprobs[w] = -1e10 
            else:
                prob = numerator / denominator
                logprobs[w] = math.log(prob)
            
        return logprobs

# Sanity check for the LM
def _check_train():
    toy = [['the','cat','sat'], ['the','dog','sat']]
    lm = TrigramLanguageModel()
    try:
        lm.train(toy)
    except NotImplementedError:
        print("Implement train() first."); return

    # Vocab + a few key counts
    assert lm.vocab == {'the','cat','dog','sat','<s>','</s>'}
    assert lm.unigram_counts['</s>'] == 2
    assert lm.bigram_counts[('sat','</s>')] == 0
    assert lm.trigram_counts[('<s>','<s>','the')] == 2
    assert lm.trigram_counts[('the','cat','sat')] == 1
    assert lm.trigram_counts[('the','dog','sat')] == 1
    print("train() sanity check passed!")

# sanity check
_check_train()

# sanity check
toy = [['the', 'cat', 'sat'], ['the', 'dog', 'sat']]
lm = TrigramLanguageModel(k=0.1)
lm.train(toy)
probs = lm.next_word_logprobs(('the', 'cat'))
assert set(probs.keys()) == lm.vocab, 'Output keys should match the vocab'
total = float(np.exp(list(probs.values())).sum())
assert abs(total - 1.0) < 1e-6, 'Probabilities must sum to one'
print('next_word_logprobs() sanity checks passed!')

train() sanity check passed!
next_word_logprobs() sanity checks passed!


## Feature extraction for NER

We will train a linear classifier to predict a NER tag for each
token.  The classifier expects a fixed set of features for each
token.  We will build a dictionary of features for each token in
the sentence.  The following information is available for each
token index `i` in a sentence:

* The token itself and its immediate context (previous and next words).
* Simple word-shape indicators (starts with an uppercase letter,
  is numeric, contains alphanumeric characters, is punctuation).
* Prefixes and suffixes (first and last two characters).
* Language model features based on your trigram LM: the log
  probability of the current word given the two previous words, and
  boolean indicators if the token is the top predicted word or
  within the top three predictions.

You can add more features if you think they might help; if you do,
please describe them in your reflection.

### Task 2: Implement `extract_features`

Complete the function below to build a feature dictionary for a token.
The required features are:

* `token` – the raw word.
* `prev` and `next` – the previous and next words (use `<s>` or `</s>`
  at sentence boundaries).
* Word shape booleans: `is_capitalised`, `is_numeric`, `is_alnum`,
  `is_punct`.
* `prefix` and `suffix` – first two and last two characters (or the
  whole token if shorter).
* Language model features: `lm_log_prob` (log probability of the
  current token), `lm_top1` (True if current token has highest
  probability in the distribution), `lm_top3` (True if current
  token is among the top three probabilities).

Return the features in a dictionary.  We will use scikit-learn's
DictVectorizer later to convert these dictionaries into numeric
feature vectors.

In [35]:
import re
from typing import Any, Dict
def extract_features(tokens: List[str], i: int, lm: TrigramLanguageModel) -> Dict[str, Any]:
    """Extract a feature dictionary for the token at position i.

    Parameters
    ----------
    tokens : List[str]
        The sentence as a list of tokens.
    i : int
        The index of the current token.
    lm : TrigramLanguageModel
        A trained trigram LM used to compute language model features.

    Returns
    -------
    Dict[str, Any]
        A feature dictionary.

    TODO: fill in this function.  Follow the instructions in the
    markdown above.
    """
    features: Dict[str, Any] = {}
    token = tokens[i]
    # Context words
    prev2 = tokens[i-2] if i >= 2 else '<s>'
    prev1 = tokens[i-1] if i >= 1 else '<s>'
    next1 = tokens[i+1] if i + 1 < len(tokens) else '</s>'

    # lexical features
    features['token'] = token
    features['prev'] = prev1
    features['next'] = next1
    
    # Shape features
    features['is_capitalised'] = token[0].isupper() if len(token) > 0 else False
    features['is_numeric'] = token.isdigit()
    features['is_alnum'] = token.isalnum()
    features['is_punct'] = all(not c.isalnum() for c in token)

    # Prefix/suffix features, depending on token's length
    if len(token) >= 2:
        features['prefix'] = token[:2]
        features['suffix'] = token[-2:]
    else:
        features['prefix'] = token
        features['suffix'] = token
    
    # Language model features
    logprobs = lm.next_word_logprobs((prev2, prev1))
    lm_lp = logprobs.get(token, float('-inf'))
    features['lm_log_prob'] = lm_lp if lm_lp > -10.0 else -10.0

    # Additional language model features (top1 and top3)
    sorted_predictions = sorted(logprobs.items(), key=lambda x: x[1], reverse=True)
    top_3_words = [word for word, prob in sorted_predictions[:3]]
    features['lm_top1'] = (token == top_3_words[0]) if top_3_words else False
    features['lm_top3'] = (token in top_3_words)
    
    return features

# Sanity Check
tokens = ['the', 'cat', 'sat']
fe = extract_features(tokens, 2, lm)  

# Lexical/boundary
assert fe['token'] == 'sat'
assert fe['prev'] == 'cat'
assert fe['next'] == '</s>'

# Shapes/prefix/suffix
assert fe['is_alnum'] is True and fe['is_punct'] is False
assert fe['prefix'] == 'sa' and fe['suffix'] == 'at'

# LM-driven features
assert fe['lm_log_prob'] > -10.0     # seen → finite, not clamped
assert fe['lm_top1'] is True         # "sat" should be the top prediction
assert fe['lm_top3'] is True
print('Feature sanity (seen trigram) passed!')

fe2 = extract_features(['hello', '...'], 1, lm)  # token="...", context=('<s>', 'hello')
assert fe2['is_punct'] is True
assert fe2['lm_log_prob'] == -10.0               # OOV → -inf → clamped to -10
assert fe2['prev'] == 'hello' and fe2['next'] == '</s>'
print('Feature sanity checks passed!')


Feature sanity (seen trigram) passed!
Feature sanity checks passed!


## Training and evaluation

Now that we have a language model and a feature extractor, we can
train a classifier.  We will use scikit-learn's logistic
regression as a simple linear classifier.  The workflow is:

1. Convert sentences into feature dictionaries and labels using
   your `extract_features` function.
2. Use `DictVectorizer` to turn these dictionaries into a sparse
   matrix.
3. Fit a logistic regression model.
4. Evaluate the model on the dev set using token-level accuracy.

You will implement helper functions for steps 1–4 and then perform
a hyperparameter sweep over several candidate values of the
smoothing constant `k`.  Choose the `k` yielding the highest dev
accuracy, retrain on the combination of train and dev data, and
finally evaluate on the test set.

### Task 3: Prepare the dataset, train the classifier and evaluate

Complete the functions below to prepare data, train a classifier and
compute accuracy.  Then perform the hyperparameter sweep and final
evaluation in the subsequent cell.

In [36]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from typing import Tuple

def prepare_dataset(sentences: List[Tuple[List[str], List[str]]], lm: TrigramLanguageModel) -> Tuple[List[Dict[str, Any]], List[str]]:
    """Convert a list of tagged sentences into features and labels.

    Each token in each sentence becomes one training example.
    """
    feat_list: List[Dict[str, Any]] = []
    label_list: List[str] = []
    for tokens, tags in sentences:
        # For each token in the sentence, extract its features and store its tag
        for i in range(len(tokens)):
            # Use the previously implemented extract_features function
            features = extract_features(tokens, i, lm)
            feat_list.append(features)
            
            # Store the corresponding BIO-style NER tag
            label_list.append(tags[i])
            
    return feat_list, label_list

def train_classifier(X_train: List[Dict[str, Any]], y_train: List[str]) -> Tuple[DictVectorizer, LogisticRegression]:
    """Fit a logistic regression classifier on dictionary features.
    Returns both the DictVectorizer and the trained classifier.
    """
    vec = DictVectorizer(sparse=True)
    X_mat = vec.fit_transform(X_train)
    clf = LogisticRegression(max_iter=100, solver='liblinear', penalty='l2', multi_class='auto')
    clf.fit(X_mat, y_train)
    return vec, clf

def evaluate(model: LogisticRegression, vectorizer: DictVectorizer, X: List[Dict[str, Any]], y_true: List[str]) -> float:
    """Compute token-level accuracy of a classifier.
    """
    # Transform X, predict and compute accuracy
    X_mat = vectorizer.transform(X)
    y_pred = model.predict(X_mat)
    acc = accuracy_score(y_true, y_pred)
    return acc

In [37]:
# Hyperparameter tuning for the smoothing constant k
candidate_ks = [0.0, 0.01, 0.1, 0.5, 1.0]
dev_accuracies = {}
for k in candidate_ks:
    print("\nTraining with k = {}".format(k))
    # Train LM
    lm = TrigramLanguageModel(k=k)
    lm.train([tokens for tokens, tags in train_data])
    # Prepare datasets
    X_tr, y_tr = prepare_dataset(train_data, lm)
    X_dev, y_dev = prepare_dataset(dev_data, lm)
    # Train classifier
    vec, clf = train_classifier(X_tr, y_tr)
    # Evaluate
    acc = evaluate(clf, vec, X_dev, y_dev)
    dev_accuracies[k] = acc
    print(f'Dev accuracy: {acc:.4f}')
# Select best k
best_k = max(dev_accuracies, key=dev_accuracies.get)
print("\nBest k based on dev set: {}".format(best_k))


Training with k = 0.0


/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Dev accuracy: 0.9851

Training with k = 0.01


/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Dev accuracy: 0.9851

Training with k = 0.1


...

Dev accuracy: 0.9970

Training with k = 0.5


/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Dev accuracy: 0.9979

Training with k = 1.0


/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Dev accuracy: 0.9979

Best k based on dev set: 0.5


In [22]:
# Final training on train+dev and evaluation on test with best k
combined = train_data + dev_data
lm_final = TrigramLanguageModel(k=best_k)
lm_final.train([tokens for tokens, tags in combined])
# Prepare data
X_comb, y_comb = prepare_dataset(combined, lm_final)
X_test, y_test = prepare_dataset(test_data, lm_final)
# Train classifier
vec_fin, clf_fin = train_classifier(X_comb, y_comb)
# Evaluate
test_acc = evaluate(clf_fin, vec_fin, X_test, y_test)
print()
print("Final test accuracy: {:.4f}".format(test_acc))

/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(



Final test accuracy: 0.9957


## Reflection

Take a few minutes to answer the following questions in this cell. We expect one to two sentences for each question.

0. Is accuracy a good metric for NER tagging, why (or why not)?

No, accuracy is generally not a good metric for NER tagging. This is because of the class imbalance program, in which the majority of tokens are labled 0 so a model can get very high accuracy by just predicting every word as 0.

1. Which smoothing constant k gave the best dev accuracy, and why?

In the experiment, the output showed that k=0.05 and k=1.0 gave the best dev accuracy at 0.9979. This is likely because smaller k values suffered from data sparsity, while the larger k values allowed a better probability distribution.

2. How does the smoothing constant affect the language model and the
   classifier performance?

Small k values keep the model close to the training data, but don't allow it to adapt to new text. Larger k values flatten the probability distribution.

3. What effect did the LM features (`lm_log_prob`, `lm_top1`,
   `lm_top3`) have on performance?  Try removing them and
   re-running the experiments.

Re-running the experiments without these LM features made the dev accuracy become constant across all values of k. The LM features must have had a negative effect at lower k values, perhaps showing that a poorly smoothed model can sometimes be worse than no model at all.

4. Suggest one extra feature you could add and hypothesise how it
   might help or hurt.

One other feature that could be added is part of speech (POS) tags. These would add tags like Noun, Verb, Adjective, etc for each word, and likely help performance by allowing the classifier to distinguish between proper and common nouns.